# Results

This notebook presents the results of the experiments conducted. The are 2 main focus areas: All the dataset with the first 10 particles per jet and no mass cut, and the dataset with a mass cut applied and also the split into 5 subsets.

The first part is collect the metrics, for that we use the `collect_metrics.py` script which gathers all the per-run/per-model metrics JSON files into a single Parquet table.

In [ ]:
# Collect metrics for the "top" task
!python3 ../scripts/collect_metrics.py --task top

Now we see the structure of the dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import polars as pl
print("All libraries imported successfully")
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))  # repo root, so `src` resolves from notebooks/
from src.utils.workspace import get_config

## Mass cut and subset results

First, lets see the structure of the dataset.

In [ ]:
# task-level metrics table written by scripts/collect_metrics.py
data = get_config("top", 0)["metrics_table_path"]
df = pd.read_parquet(data, engine='pyarrow')
df.columns

This shows the first 20 rows of the dataset.

In [ ]:
df.head(20)

In [ ]:
df.info()

Now let's see the statistical summary of the dataset, classified by the 'model' column, and only for the seeds 10, 11, 12, 13, and 14.

In [ ]:
# statistical description of the dataset by 'model' and selected seeds
metrics = ['Test Loss','Test F1 Score']

df.loc[
    df['seed'].isin([10, 11, 12, 13, 14]),
    ['model', *metrics]
].groupby('model')[metrics].describe()

In [ ]:
# statistical description of the dataset by 'model' and selected seeds
metrics = ['Test Accuracy', 'Test AUC']

df.loc[
    df['seed'].isin([10, 11, 12, 13, 14]),
    ['model', *metrics]
].groupby('model')[metrics].describe()

In [ ]:
# Agrupamos por semillas y por modelo, solo considerando las réplicas con corte de masa y n_subsets > 1
# (excluye la corrida de dataset completo y las legacy; se selecciona por columnas, no por semilla)
replicates = df[(df['apply_mass_cut'] == True) & (df['n_subsets'] > 1)]
df_grouped = replicates.groupby(['seed', 'model']).mean(numeric_only=True).reset_index()
#df_grouped.head(10)
# calculamos el promedio y desviación estandar de cada columna numérica agrupando modelo y calculando por semilla
df_stats = df_grouped.groupby('model').agg(['mean', 'std']).reset_index()
df_stats.head(10)

In [ ]:
#grafica de accuracy con barra de error 
plt.figure(figsize=(10, 6))
plt.errorbar(df_stats['model'], df_stats['Test Accuracy']['mean'], 
            yerr=df_stats['Test Accuracy']['std'], fmt='o', capsize=5) 
# rotamos label de modelo 45 grados
plt.xticks(rotation=45)
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.title('Accuracy with Error Bars by Model')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
#grafica de accuracy con barra de error 
plt.figure(figsize=(10, 6))
plt.errorbar(df_stats['model'], df_stats['Test AUC']['mean'], 
            yerr=df_stats['Test AUC']['std'], fmt='o', capsize=5) 
# rotamos label de modelo 45 grados
plt.xticks(rotation=45)
plt.xlabel('Model')
plt.ylabel('AUC')
plt.title('AUC with Error Bars by Model')
plt.grid(True, linestyle='--', alpha=0.7)
plt.ylim(0.65,0.85)
plt.show()

#### 3) Fenómeno de Colapso de Calibración (Miscalibration)
Un fenómeno recurrente documentado en nuestros experimentos es el llamado **colapso de la matriz de confusión**. Muchos modelos cuánticos no entrenados (e incluso algunos entrenados) reportan exhaustividad (Recall) $\approx 1.0$ y precisión $\approx 0.5$, clasificando casi todos los eventos como positivos.

**¿Por qué sucede esto?**
El gate de lectura devuelve valores esperados físicos estrictamente acotados en el rango $[-1, 1]$. Al aplicar la función sigmoide:

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

los límites del logit $[-1, 1]$ se mapean exclusivamente dentro del intervalo $[0.269, 0.731]$. Debido a esto:
* Una muy leve desviación o sesgo estático (offset) heredado de las neuronas clásicas en el warm-start es suficiente para inclinar permanentemente los logits por encima o debajo del umbral de decisión rígido de $0.5$.
* Sin embargo, el **ROC-AUC (Área bajo la curva ROC)**, que es una métrica de ordenamiento independiente del umbral de corte, escala monótonamente hacia arriba durante el entrenamiento (de $0.70$ inicial a más de $0.74$). Esto demuestra que el circuito cuántico **sí está aprendiendo a separar las clases**, por lo que evaluar los modelos basándose puramente en Exactitud (Accuracy) resulta engañoso frente al análisis del ROC-AUC.

2. **El "Arranque en Caliente" es Superior a la Búsqueda Ciega:** Inicializar circuitos cuánticos mediante ajustes de interpolación como polinomios de Chebyshev o transformadas de Fourier simplifica los paisajes de pérdida (loss landscapes), permitiendo que un VQC tenga señal predictiva real incluso sin entrenamiento previo.
3. **El Desafio Metrológico Cuántico (Futuras Mitigaciones):**
   * **Umbrales Optimizados:** Para solucionar el colapso de calibración cuántica (donde el sigmoide comprime las salidas en $[0.269, 0.731]$), se propone sustituir el umbral fijo de 0.50 por un umbral óptimo dinámico derivado de la curva ROC-Validation (usando el índice de *Youden's J*).
   * **Escalamiento No Lineal de Logits:** Implementar factores de multiplicación y offsets sintonizables en la capa de salida cuántica antes de aplicar la función de activación sigmoide resolvería la compresión de probabilidades, permitiendo al circuito emitir decisiones con mayor confianza (acercándose a $[0, 1]$).
   * **Hardware NISQ y Mitigación de Ruido:** El modelo incorpora simuladores ruidosos basados en perfiles reales (como `FakeManilaV2`). El desarrollo de la técnica de mitigación de errores (zero-noise extrapolation) guiará la portabilidad del modelo sobre ordenadores cuánticos superconductores reales de IBM Quantum.

## No mass cut and first 10 particles per jet